# Praktikum: Klasifikasi Citra Emosi Wajah dengan CNN + Dashboard Streamlit
Versi VS Code (lokal). Pilih kernel Python di pojok kanan atas, lalu Run All atau jalankan cell satu per satu.
Semua hasil disimpan di folder `output/` di samping notebook ini.

## Langkah 0 - Import & konfigurasi

In [ ]:
%pip install -q kagglehub

import os, json, random, shutil
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)

IMG_SIZE = (48, 48)
BATCH = 64
EPOCHS = 30
OUT_DIR = 'output'
os.makedirs(OUT_DIR, exist_ok=True)

## Langkah 1 - Ambil dataset dari Kaggle

In [ ]:
# Opsi A (otomatis): unduh lewat kagglehub (butuh internet; kalau diminta login Kaggle, ikuti instruksinya)
import kagglehub
path = kagglehub.dataset_download("fahadullaha/facial-emotion-recognition-dataset")
# Opsi B (manual): kalau sudah download zip dari Kaggle dan diekstrak, pakai path-nya:
# path = r"C:\Users\NAMA\Downloads\facial-emotion-recognition-dataset"
print("Dataset di:", path)

# Lihat struktur folder (2 level)
for root, dirs, files in os.walk(path):
    depth = root[len(path):].count(os.sep)
    if depth <= 2:
        print('  ' * depth + os.path.basename(root) + f'/  ({len(files)} file, {len(dirs)} folder)')

In [ ]:
# PENTING: DATA_DIR harus folder yang ISINYA langsung folder-folder kelas
# (angry, happy, sad, ...). Kalau ada folder train/test, pilih 'train'.
# Cek output cell di atas, lalu ubah kalau perlu.
def find_class_root(p):
    for root, dirs, files in os.walk(p):
        if 'train' in dirs:
            return os.path.join(root, 'train')
    return p

DATA_DIR = find_class_root(path)
# DATA_DIR = os.path.join(path, 'nama_folder_yang_benar')   # <- uncomment kalau salah
print("DATA_DIR:", DATA_DIR)
print("Isi:", sorted(os.listdir(DATA_DIR)))

## Langkah 2 - EDA (distribusi kelas, contoh gambar, ukuran & mode warna)

In [ ]:
exts = ('.jpg', '.jpeg', '.png', '.bmp')
class_names = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])
counts = {c: len([f for f in os.listdir(os.path.join(DATA_DIR, c)) if f.lower().endswith(exts)])
          for c in class_names}
NUM_CLASSES = len(class_names)
total = sum(counts.values())
print("Kelas:", class_names)
print("Jumlah per kelas:", counts)
print("Total gambar:", total)

ratio = max(counts.values()) / max(1, min(counts.values()))
print(f"Rasio kelas terbesar : terkecil = {ratio:.1f}x ->",
      "IMBALANCE (pakai class weight)" if ratio > 1.5 else "relatif seimbang")

# Distribusi
plt.figure(figsize=(8, 4))
bars = plt.bar(counts.keys(), counts.values(), color='steelblue')
plt.bar_label(bars)
plt.title('Distribusi Jumlah Gambar per Kelas'); plt.ylabel('Jumlah'); plt.xticks(rotation=30)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/1_class_distribution.png', dpi=120); plt.show()

# Ukuran & mode warna (sampel 300 gambar)
sizes, modes = set(), set()
all_files = [os.path.join(DATA_DIR, c, f) for c in class_names
             for f in os.listdir(os.path.join(DATA_DIR, c)) if f.lower().endswith(exts)]
for fp in random.sample(all_files, min(300, len(all_files))):
    with Image.open(fp) as im:
        sizes.add(im.size); modes.add(im.mode)
print("Ukuran gambar (sampel):", sizes)
print("Mode warna (sampel)   :", modes, "(L = grayscale, RGB = berwarna)")

# Contoh gambar: 5 per kelas
fig, axes = plt.subplots(NUM_CLASSES, 5, figsize=(10, 2 * NUM_CLASSES))
for r, c in enumerate(class_names):
    fl = [f for f in os.listdir(os.path.join(DATA_DIR, c)) if f.lower().endswith(exts)]
    for k, f in enumerate(random.sample(fl, min(5, len(fl)))):
        axes[r, k].imshow(Image.open(os.path.join(DATA_DIR, c, f)).convert('L'), cmap='gray')
        axes[r, k].axis('off')
        if k == 0:
            axes[r, k].set_title(c, loc='left', fontsize=10)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/1_sample_images.png', dpi=100); plt.show()

## Langkah 3 - Pra-pemrosesan (resize 48x48, grayscale, split 80/20, augmentasi, one-hot)
Normalisasi 0-1 dilakukan di dalam model lewat layer `Rescaling`, jadi saat prediksi di Streamlit
gambar cukup dikirim apa adanya (0-255).

In [ ]:
common = dict(validation_split=0.2, seed=SEED, image_size=IMG_SIZE, batch_size=BATCH,
              label_mode='categorical', color_mode='grayscale', class_names=class_names)
train_raw = tf.keras.utils.image_dataset_from_directory(DATA_DIR, subset='training', **common)
val_raw = tf.keras.utils.image_dataset_from_directory(DATA_DIR, subset='validation', **common)

n_train = int(tf.data.experimental.cardinality(train_raw)) 
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_raw.cache().prefetch(AUTOTUNE)
val_ds = val_raw.cache().prefetch(AUTOTUNE)

# Class weight untuk mengatasi imbalance
class_weight = {i: total / (NUM_CLASSES * counts[c]) for i, c in enumerate(class_names)}
print("Class weight:", {class_names[i]: round(w, 2) for i, w in class_weight.items()})

n_val_images = sum(int(x.shape[0]) for x, _ in val_raw)
n_train_images = total - n_val_images
print(f"Train: {n_train_images} | Validasi: {n_val_images}")

json.dump(class_names, open(f'{OUT_DIR}/class_names.json', 'w'))
json.dump({'counts': counts, 'total': total, 'n_train': n_train_images, 'n_val': n_val_images,
           'img_size': list(IMG_SIZE), 'imbalance_ratio': ratio},
          open(f'{OUT_DIR}/dataset_info.json', 'w'))

## Langkah 4 - Bangun model CNN

In [ ]:
augment = models.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.1, 0.1),
], name='augmentation')

inputs = layers.Input(shape=IMG_SIZE + (1,))
x = augment(inputs)
x = layers.Rescaling(1. / 255)(x)

filters = [32, 64, 128, 256]
for i, f in enumerate(filters):
    x = layers.Conv2D(f, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    last = (i == len(filters) - 1)
    x = layers.Conv2D(f, 3, padding='same', activation='relu',
                      name='last_conv' if last else None)(x)   # dipakai Grad-CAM
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.25)(x)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = models.Model(inputs, outputs, name='emotion_cnn')
model.compile(optimizer=Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

## Langkah 4b - Training

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True),
    ModelCheckpoint(f'{OUT_DIR}/best_emotion_cnn.keras', monitor='val_accuracy', save_best_only=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-6),
]
hist = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
                 class_weight=class_weight, callbacks=callbacks)
history = {k: [float(v) for v in vals] for k, vals in hist.history.items()}
json.dump(history, open(f'{OUT_DIR}/history.json', 'w'))

## Langkah 5 & 6 - Evaluasi + visualisasi

In [ ]:
loss, acc = model.evaluate(val_ds, verbose=0)
print(f"Akurasi validasi: {acc*100:.2f}% | Loss: {loss:.4f}")

# Kurva akurasi & loss
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, (m, t) in zip(ax, [('accuracy', 'Accuracy'), ('loss', 'Loss')]):
    a.plot(history[m], 'o-', label=f'Train {t}')
    a.plot(history[f'val_{m}'], 's-', label=f'Val {t}')
    a.set_title(f'Training vs Validation {t}'); a.set_xlabel('Epoch'); a.legend(); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/2_training_curves.png', dpi=120); plt.show()

# Kumpulkan label & prediksi dalam SATU kali iterasi (urutan val_ds diacak tiap iterasi)
y_true, y_pred, imgs_keep, probs_keep = [], [], [], []
for xb, yb in val_ds:
    p = model.predict(xb, verbose=0)
    y_true.extend(np.argmax(yb.numpy(), axis=1))
    y_pred.extend(np.argmax(p, axis=1))
    need = 16 - len(imgs_keep)
    if need > 0:
        imgs_keep.extend(xb[:need].numpy()); probs_keep.extend(p[:need])
y_true, y_pred = np.array(y_true), np.array(y_pred)
labels = list(range(NUM_CLASSES))

print(classification_report(y_true, y_pred, labels=labels, target_names=class_names, digits=3, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=labels)
cm_pct = cm / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm_pct, cmap='Blues', vmin=0, vmax=1); fig.colorbar(im, label='Proporsi')
ax.set_xticks(labels); ax.set_xticklabels(class_names, rotation=45, ha='right')
ax.set_yticks(labels); ax.set_yticklabels(class_names)
for i in labels:
    for j in labels:
        ax.text(j, i, cm[i, j], ha='center', va='center', color='white' if cm_pct[i, j] > .5 else 'black')
ax.set_xlabel('Prediksi'); ax.set_ylabel('Kelas sebenarnya'); ax.set_title('Confusion Matrix')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/3_confusion_matrix.png', dpi=120); plt.show()

# Metrik ringkas -> metrics.json (dipakai dashboard)
pm, rm, fm, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
pw, rw, fw, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
metrics = {
    'accuracy': float(accuracy_score(y_true, y_pred)),
    'macro': {'precision': float(pm), 'recall': float(rm), 'f1': float(fm)},
    'weighted': {'precision': float(pw), 'recall': float(rw), 'f1': float(fw)},
    'per_class': classification_report(y_true, y_pred, labels=labels, target_names=class_names,
                                       output_dict=True, zero_division=0),
    'confusion_matrix': cm.tolist(),
    'class_names': class_names,
}
json.dump(metrics, open(f'{OUT_DIR}/metrics.json', 'w'))

# Contoh prediksi
fig, axes = plt.subplots(3, 4, figsize=(10, 8))
for i, a in enumerate(axes.flatten()):
    img = imgs_keep[i].squeeze() / 255.
    pr = int(np.argmax(probs_keep[i])); tr = int(y_true[i])
    a.imshow(img, cmap='gray'); a.axis('off')
    a.set_title(f"Pred: {class_names[pr]} ({probs_keep[i][pr]*100:.0f}%)\nAsli: {class_names[tr]}",
                color='green' if pr == tr else 'red', fontsize=9)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/4_sample_predictions.png', dpi=120); plt.show()

## Langkah 6.3 - Feature importance dengan Grad-CAM
Area terang/merah = bagian wajah yang paling memengaruhi keputusan model.

In [ ]:
def gradcam(model, img, class_idx=None):
    grad_model = tf.keras.Model(model.inputs, [model.get_layer('last_conv').output, model.output])
    with tf.GradientTape() as tape:
        conv, preds = grad_model(img[None].astype('float32'), training=False)
        if class_idx is None:
            class_idx = int(tf.argmax(preds[0]))
        score = preds[:, class_idx]
    grads = tape.gradient(score, conv)
    w = tf.reduce_mean(grads, axis=(0, 1, 2))
    cam = tf.maximum(tf.reduce_sum(conv[0] * w, axis=-1), 0)
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    cam = tf.image.resize(cam[..., None], IMG_SIZE).numpy().squeeze()
    return cam, class_idx

fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for k in range(6):
    img = imgs_keep[k]
    cam, ci = gradcam(model, img)
    axes[0, k].imshow(img.squeeze(), cmap='gray'); axes[0, k].axis('off')
    axes[0, k].set_title(f'Asli: {class_names[int(y_true[k])]}', fontsize=9)
    axes[1, k].imshow(img.squeeze(), cmap='gray'); axes[1, k].imshow(cam, cmap='jet', alpha=.45)
    axes[1, k].axis('off'); axes[1, k].set_title(f'Pred: {class_names[ci]}', fontsize=9)
plt.suptitle('Grad-CAM: area yang memengaruhi prediksi', fontweight='bold')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/5_gradcam.png', dpi=120); plt.show()

## Langkah 4.5 - Simpan model

In [ ]:
model.save(f'{OUT_DIR}/final_emotion_cnn.keras')
loaded = tf.keras.models.load_model(f'{OUT_DIR}/final_emotion_cnn.keras')
print("Model tersimpan & berhasil di-load ulang.")

print("Selesai. Hasil ada di:", os.path.abspath(OUT_DIR))
print("Taruh app.py sejajar dengan folder 'output', lalu jalankan: streamlit run app.py")